# Quickstart: one evaluation, end to end

Builds one candidate, evaluates it on `BENCHMARK`, and asserts that the client minted a trace
id, the candidate compiled to url4, and the score sits in the hand-set `SCORE_BAND`. The
`workbench-result` cell reports the trace ids right after the evaluation, before any
assertion, so a failing check still leaves them in the evidence; the workbench then requires
each one to appear in the engine and aigateway logs.

On `replay` the recipe is the golden ifeval recipe (served from the recorded cache, no spend);
on `local` it is the authored open-source fusion below, run with your OpenRouter key.

In [ ]:
ENGINE_URL = "http://127.0.0.1:9108"
SCOREBOARD_URL = None
RECIPE = "authored"  # "golden" on replay: the exact recipe the ifeval tape recorded
BENCHMARK = "ifeval"
LIMIT = 1
SCORE_BAND = [0.0, 1.0]  # hand-set sanity band (owner decision 2026-10-16)
EXPECT_CACHED = False

In [ ]:
import json
import os

import screamingface as sf

client = (
    sf.Client(engine_url=ENGINE_URL, scoreboard_url=SCOREBOARD_URL)
    if SCOREBOARD_URL
    else sf.Client(engine_url=ENGINE_URL)
)
if os.environ.get("OPENROUTER_API_KEY") and RECIPE == "authored":
    client.connect("openrouter", api_key=os.environ["OPENROUTER_API_KEY"])

In [ ]:
if RECIPE == "golden":
    candidate = sf.CorrectiveLoop(
        [
            sf.Model(
                "openrouter/mistralai/ministral-3b-2512",
                prompt=(
                    "Answer the request accurately and completely. Follow every "
                    "instruction and formatting constraint in the request."
                ),
                params={"max_tokens": 8192, "temperature": 0.0},
            ),
            sf.Model(
                "openrouter/microsoft/phi-4",
                prompt=(
                    "Answer the request accurately and completely. Follow every "
                    "instruction and formatting constraint in the request."
                ),
                params={"max_tokens": 8192, "temperature": 0.0},
            ),
        ],
        judge=sf.Model(
            "openrouter/deepseek/deepseek-v4-flash",
            prompt=(
                "Produce one final answer to the original request from the panel "
                "drafts. Preserve every instruction and formatting constraint."
            ),
            params={"max_tokens": 8192, "temperature": 0.0},
        ),
        max_rounds=3,
    )
else:
    ANSWER = (
        "Answer the request accurately and completely. "
        "Follow every instruction and formatting constraint in the request."
    )
    SYNTH = (
        "Produce the single best answer to the original request by combining the panel "
        "drafts. Preserve every instruction and formatting constraint stated in the "
        "original prompt."
    )
    candidate = sf.Fusion(
        [
            sf.Model("openrouter/deepseek/deepseek-v4-pro", prompt=ANSWER),
            sf.Model("openrouter/qwen/qwen3-coder", prompt=ANSWER),
        ],
        name="best_open_source",
        synthesizer=sf.Model("openrouter/z-ai/glm-5.2", prompt=SYNTH),
    )
candidate

In [ ]:
report = client.evaluate(candidate, benchmark=BENCHMARK, limit=LIMIT, progress=False)

In [ ]:
print(
    "WORKBENCH_RESULT="
    + json.dumps(
        {
            "trace_ids": [c.trace_id for c in report.candidates if c.trace_id],
            "run_ids": [str(c.run_id) for c in report.candidates],
            "candidates": {
                c.name: {
                    "score": c.score,
                    "cache_hits": c.cache_hits,
                    "cache_saved_cost_usd": str(c.cache_saved_cost_usd),
                }
                for c in report.candidates
            },
        }
    )
)

In [ ]:
(result,) = list(report.candidates)
assert result.trace_id, "the client minted no trace id (OME-967)"
assert result.url4, "the candidate compiled to no url4"
assert result.score is not None and SCORE_BAND[0] <= result.score <= SCORE_BAND[1], result.score
if EXPECT_CACHED:
    assert result.cache_hits > 0, "expected a warm cache on this target"
report

In [ ]:
client.close()